# Lab 07｜把一次纠正变成下一次能用的方法

同样是支付延迟，新事故可能有不同原因。把旧经历提炼成 Skill，究竟保留了方法，还是偷偷保存了旧答案？

**本例范围**：下面现场生成两轮教学对话，不是正文完整排查经历的回放。本例直接请求模型提炼候选 `SKILL.md`，未调用系统 `$skill-creator` 或它的脚本；正文中的 Skill Creator 工作流需要另行实践。这里观察经历提炼和 `SkillInput` 使用，并加一个无 Skill 对照。

## 核心代码：从本次经历提炼候选方法，再交给新任务

> 本讲重点看下面这 10 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 2 节，第 7 个单元格，第 1 至 7 行：**

```python
extractor = lab.thread()
extracted = run(extractor,
    "请从下面这段教学对话提炼一个可供新任务使用的中文 Skill。写清适用场景和可复用方法，"
    "尤其说明纠正怎样改变判断；不要把一次事故的具体答案当作通用结论。"
    "只输出 SKILL.md 原文，YAML 头含 name: latency-investigation 与中文 description，正文不超过 700 字，"
    "不加代码围栏，不调用工具。\n对话：\n" + json.dumps(messages,ensure_ascii=False))
skill_text = extracted.final_response.strip()
```

`messages` 来自本次教学对话；这次调用要求模型提炼方法而非保存旧事故答案，随后取出候选正文。

**第 2 节，第 7 个单元格，第 9 行：**

```python
skill_path = save_candidate(skill_text)
```

`save_candidate()` 保存候选文件，格式符合要求不代表方法已经有效。

**第 3 节，第 9 个单元格，第 8 至 9 行：**

```python
skilled_thread = lab.thread()
applied = run(skilled_thread,[SkillInput(name="latency-investigation",path=str(skill_path)),TextInput(new_task)])
```

新建 Thread，再通过 `SkillInput` 把候选方法与当前任务一起交给模型；前面的无 Skill 运行用于对照。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `GLM_API_KEY`；未设置时弹出隐藏输入框。模型由 GLM-5.2 提供，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、协议适配、健康检查和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0", "litellm": "1.101.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(("litellm[proxy]" if package == "litellm" else package) + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex", "litellm"))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("GLM_API_KEY", "").strip():
    os.environ["GLM_API_KEY"] = getpass.getpass("GLM_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "07-skill-extraction")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work
from openai_codex import SkillInput, TextInput

def save_candidate(text):
    import yaml
    parts = text.strip().split("---",2)
    if len(parts)!=3 or parts[0].strip() or not parts[2].strip():
        raise ValueError("候选文件缺少完整 YAML 头或正文，请查看原始提炼输出后再试。")
    header = yaml.safe_load(parts[1])
    if not isinstance(header,dict) or header.get("name")!="latency-investigation" or not isinstance(header.get("description"),str) or not header["description"].strip():
        raise ValueError("候选文件需要 name: latency-investigation 与非空 description。")
    # 放在自动发现目录之外，避免无 Skill 对照意外读到描述或正文。
    path = WORK / "candidates" / "latency-investigation" / "SKILL.md"
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text,encoding="utf-8")
    return path

print("公共准备完成。")


## 1. 留下一段带纠正的教学对话｜必读必跑

旧猜测来自输入中的巡检记录，后续采样否定它。两次回答都由模型实际生成。只把本次消息留在内存中供下一步提炼，不另写无人读取的经历文件。


In [ ]:
import json
from openai_codex import SkillInput, TextInput
investigation = lab.thread()
initial = run(investigation,
    "教学案例 OLD-184：支付接口延迟升高。旧巡检记录猜测 CPU 饱和，但还没有同时间窗指标。"
    "请给出只读核对计划，区分猜测和事实。不要调用工具，不得建议直接重启。回答不超过 160 字。")
corrected = run(investigation,
    "补充采样：02:07 CPU 为 21%，连接占用 60/60，连接池等待 3600ms，下游耗时 110ms。"
    "变更单显示 01:40 连接池上限由 200 调到 60。CPU 饱和猜测已被排除。"
    "请修正判断，保留尚未证明的因果边界，不执行变更或重启，不调用工具。回答不超过 200 字。")
trace = investigation.read(include_turns=True).model_dump(mode="json", by_alias=True)
# 只保存这次教学对话的消息，不把基础配置或本机路径写进 Skill。
messages = []
for turn in trace["thread"]["turns"]:
    for item in turn["items"]:
        if item.get("type") in ("userMessage", "agentMessage"):
            messages.append(item)
table(["对话阶段", "模型实际回答"],[("初步计划",initial.final_response),("收到纠正后",corrected.final_response)])


## 2. 请模型从经历提炼方法｜必读必跑

只规定文件接口和提炼目标，不预先替它写调查方法。生成后先读 Skill 正文：**哪些方法来自这次纠正，哪些是提炼时额外补充的要求？** 对额外内容判断是否合理，不把模型写下的每条要求都当成这段经历已经支持的结论。


In [ ]:
extractor = lab.thread()
extracted = run(extractor,
    "请从下面这段教学对话提炼一个可供新任务使用的中文 Skill。写清适用场景和可复用方法，"
    "尤其说明纠正怎样改变判断；不要把一次事故的具体答案当作通用结论。"
    "只输出 SKILL.md 原文，YAML 头含 name: latency-investigation 与中文 description，正文不超过 700 字，"
    "不加代码围栏，不调用工具。\n对话：\n" + json.dumps(messages,ensure_ascii=False))
skill_text = extracted.final_response.strip()
display(Markdown("### 本次提炼出的 Skill\n\n" + skill_text))
skill_path = save_candidate(skill_text)
details("扩展：格式检查", "已核对 YAML 头、调用名称、非空描述及正文；这些检查不判断方法质量。")


## 3. 同一新任务：无 Skill 与有 Skill｜必读必跑

两次运行都使用全新 Thread、相同现场和相同任务提示。有 Skill 的一次额外收到刚才的候选方法。提示只交代目标，调查方法留给 Skill。

对照回答看：它们怎样处理旧猜测、当前数据、下一步与不确定性？若两次没有明显差别，就说明本次对照没有呈现清晰增量；不能只因“有 Skill 的运行完成了”就宣布方法有效。


In [ ]:
new_task = """新案例：同一采样窗口内，CPU 23%，连接占用 18/200，连接等待 9ms，
下游调用耗时 4200ms，接口总耗时 4300ms。没有提供配置变更记录。
请给出简短判断与下一步建议。不要调用工具。"""
plain_start = len(lab.requests)
plain_thread = lab.thread()
plain = run(plain_thread,new_task)
skilled_start = len(lab.requests)
skilled_thread = lab.thread()
applied = run(skilled_thread,[SkillInput(name="latency-investigation",path=str(skill_path)),TextInput(new_task)])
table(["同一新任务", "实际回答"],[("未加载 Skill",plain.final_response),("加载候选 Skill",applied.final_response)])
plain_request = lab.request_text(plain_start)
skilled_request = lab.request_text(skilled_start)
encoded_skill = json.dumps(skill_text,ensure_ascii=False)[1:-1]
table(["实际模型输入", "无 Skill", "有 Skill"], [
    ("候选 Skill 正文", "存在" if encoded_skill in plain_request else "未装入", "存在" if encoded_skill in skilled_request else "未见"),
    ("旧事故教学对话", "存在" if "OLD-184" in plain_request else "未带入", "存在旧编号，请检查候选是否照搬" if "OLD-184" in skilled_request else "未带入旧编号"),
])
details("扩展：供逐条追溯的原始教学消息",json.dumps(messages,ensure_ascii=False,indent=2))


## 继续观察：初始化 Skill 目录｜扩展/可选

默认提炼流程直接生成候选 `SKILL.md`，没有调用系统 Skill Creator。若本机已安装 Codex 的 `skill-creator`，可启用下格的 `RUN_CREATOR_INIT`，只在本次工作目录中初始化另一个演示骨架，不安装个人 Skill，也不替换前面提炼的候选。

命令的两个路径由下格确定，含空格时须保留引号：

```bash
python3 "$SKILL_CREATOR_ROOT/scripts/init_skill.py" legacy-payment-investigation \
  --path "$SKILL_DEMO_DIR" \
  --resources scripts,references \
  --interface 'short_description=调查 legacy-payment 模块告警，核对日志、指标与运行配置'
```

`generate_openai_yaml.py` 要求 `short_description` 长度为 25—64 个字符。这里的说明为 35 个字符。初始化只生成骨架，不能验证提炼方法或新事故判断；完成后仍须执行最后的清理格。


In [ ]:
RUN_CREATOR_INIT = False
if RUN_CREATOR_INIT:
    creator_root = Path(os.environ.get("SKILL_CREATOR_ROOT",
                        str(Path.home() / ".codex/skills/.system/skill-creator")))
    init_script = creator_root / "scripts/init_skill.py"
    if not init_script.is_file():
        raise FileNotFoundError("未找到 Skill Creator；请把 SKILL_CREATOR_ROOT 设置为本机实际安装目录。")
    description = "调查 legacy-payment 模块告警，核对日志、指标与运行配置"
    if not 25 <= len(description) <= 64:
        raise ValueError("short_description 必须为 25—64 个字符。")
    demo_dir = WORK / "creator-demo"
    command = [sys.executable, str(init_script), "legacy-payment-investigation",
               "--path", str(demo_dir), "--resources", "scripts,references",
               "--interface", "short_description=" + description]
    print("SKILL_CREATOR_ROOT：", creator_root)
    print("SKILL_DEMO_DIR：", demo_dir)
    subprocess.check_call(command)
    print("已初始化演示骨架；不是前面提炼的候选 Skill，也未安装到个人目录。")


In [ ]:
# 比较和逐条阅读结束后关闭；本次不会安装或修改个人 Skill。
lab.close()
